# Model Registry

## What is a Model Registry?

A **model registry** is a centralized repository that stores, versions, and manages ML models throughout their lifecycle from experimentation to production retirement.

### Why It Matters

| Without Registry | With Registry |
|-----------------|---------------|
| Models saved as random files | Versioned, named, discoverable |
| No audit trail | Full lineage: data → run → model → deploy |
| Manual deployment | Automated promotion pipelines |
| No rollback | One-command rollback to previous version |

### Model Lifecycle Stages

```
Experiment → [None] → Staging → Production → Archived
```

### Semantic Versioning for ML Models

```
MAJOR.MINOR.PATCH
  │      │     └── Bug fix, retrain with same data/algo
  │      └──────── New feature, architecture tweak
  └─────────────── Breaking change, new task/schema
```

## 1. MLflow Model Registry

MLflow's registry provides stage-based model lifecycle management.

### Key Concepts
- **Registered Model**: a named model (e.g. `fraud-detector`)
- **Model Version**: each `mlflow.register_model()` call creates a new version
- **Stage**: `None → Staging → Production → Archived`
- **Alias**: human-readable pointer e.g. `champion`, `challenger`

In [1]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Setup
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # local SQLite backend
mlflow.set_experiment("registry-demo")

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Train and log
with mlflow.start_run() as run:
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))

    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("accuracy", acc)
    mlflow.sklearn.log_model(model, artifact_path="model")

    run_id = run.info.run_id
    print(f"Run ID: {run_id} | Accuracy: {acc:.4f}")

# Register the model
model_uri = f"runs:/{run_id}/model"
registered = mlflow.register_model(model_uri=model_uri, name="breast-cancer-classifier")
print(f"Registered: version={registered.version}, stage={registered.current_stage}")

2026/06/19 16:59:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/19 17:00:32 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.1+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.12.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Registered model 'breast-cancer-classifier' already exists. Creating a new version of this model...
2026/06/19 17:00:32 WARNING mlflow.tracking._model_registry.fluent: Run with id b0d98c8f5974445eb244495b30c173b6 has no artifacts at artifact path 'model', registering model based on models:/m-99d283e0e38c4dbd859f5c5974dab863 instead


Run ID: b0d98c8f5974445eb244495b30c173b6 | Accuracy: 0.9650
Registered: version=2, stage=None


Created version '2' of model 'breast-cancer-classifier'.


In [2]:
client = MlflowClient()

# Transition to Staging
client.transition_model_version_stage(
    name="breast-cancer-classifier",
    version=registered.version,
    stage="Staging",
    archive_existing_versions=False
)

# After validation, promote to Production
client.transition_model_version_stage(
    name="breast-cancer-classifier",
    version=registered.version,
    stage="Production",
    archive_existing_versions=True  # archive old production version
)

# Set alias (MLflow 2.x+)
client.set_registered_model_alias(
    name="breast-cancer-classifier",
    alias="champion",
    version=registered.version
)

# Load by alias
champion = mlflow.sklearn.load_model("models:/breast-cancer-classifier@champion")
print(f"Champion model loaded: {champion}")

# Load by stage
prod_model = mlflow.sklearn.load_model("models:/breast-cancer-classifier/Production")
print(f"Production model loaded: {prod_model}")

/tmp/ipykernel_158994/3799800591.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
/tmp/ipykernel_158994/3799800591.py:12: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Champion model loaded: RandomForestClassifier(random_state=42)
Production model loaded: RandomForestClassifier(random_state=42)


In [3]:
# List all versions
for mv in client.search_model_versions("name='breast-cancer-classifier'"):
    print(f"Version {mv.version} | Stage: {mv.current_stage} | Run: {mv.run_id[:8]}")

# Add description
client.update_registered_model(
    name="breast-cancer-classifier",
    description="Breast cancer classifier. Input: 30 numeric features. Output: binary."
)

# Add tags
client.set_registered_model_tag("breast-cancer-classifier", "team", "ml-platform")
client.set_model_version_tag("breast-cancer-classifier", registered.version, "validated", "true")

# Archive (retire)
# client.transition_model_version_stage(name="breast-cancer-classifier", version="1", stage="Archived")

Version 2 | Stage: Production | Run: b0d98c8f
Version 1 | Stage: Archived | Run: ee1ba117


## 2. Hugging Face Hub as Registry

HF Hub is the de facto registry for transformer models. Every model is a git repo with LFS for weights.

In [4]:
from huggingface_hub import HfApi  # Repository removed in recent huggingface_hub
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Push model to Hub
# model.push_to_hub("username/my-sentiment-model", private=True)
# tokenizer.push_to_hub("username/my-sentiment-model")

# Create model card programmatically
from huggingface_hub import ModelCard

card_content = """
---
language: en
license: apache-2.0
tags:
  - text-classification
  - sentiment
metrics:
  - accuracy
---

# My Sentiment Model

Fine-tuned DistilBERT on SST-2. Accuracy: 0.921
"""

# Version with git tags
api = HfApi()
# api.create_tag("username/my-model", tag="v1.0.0", message="Production release")

# Load specific version by tag
# model = AutoModelForSequenceClassification.from_pretrained(
#     "username/my-model", revision="v1.0.0"
# )

print("HF Hub patterns demonstrated (commented out to avoid network calls)")

HF Hub patterns demonstrated (commented out to avoid network calls)


## 3. Weights & Biases Model Registry

In [5]:
# W&B Artifacts for model versioning
import wandb
import joblib

# Log model as artifact
# with wandb.init(project="registry-demo") as run:
#     # Train model...
#     joblib.dump(model, "model.pkl")

#     artifact = wandb.Artifact(
#         name="breast-cancer-classifier",
#         type="model",
#         description="RF classifier, 100 trees",
#         metadata={"accuracy": 0.965, "n_estimators": 100}
#     )
#     artifact.add_file("model.pkl")
#     run.log_artifact(artifact)

# Link to W&B Model Registry
# run.link_artifact(artifact, target_path="registry/breast-cancer-classifier:production")

# Download artifact
# api = wandb.Api()
# artifact = api.artifact("entity/project/breast-cancer-classifier:latest")
# artifact.download()

print("W&B patterns demonstrated (commented out to avoid auth requirement)")

W&B patterns demonstrated (commented out to avoid auth requirement)


## 4. BentoML Model Store

In [6]:
import bentoml
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = RandomForestClassifier(n_estimators=100).fit(X_train, y_train)

# Save to BentoML local store
saved = bentoml.sklearn.save_model(
    "breast_cancer_classifier",
    model,
    signatures={"predict": {"batchable": True}},
    labels={"stage": "production", "accuracy": "0.965"},
    metadata={"framework": "sklearn", "n_estimators": 100}
)
print(f"Saved: {saved.tag}")

# Load by tag
loaded = bentoml.sklearn.load_model("breast_cancer_classifier:latest")
print(f"Loaded model type: {type(loaded)}")

# List all versions
for m in bentoml.models.list():
    print(f"  {m.tag} labels: {m.info.labels}")

Saved: breast_cancer_classifier:pzvvdz3l22iai5dy
Loaded model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
  breast_cancer_classifier:pzvvdz3l22iai5dy labels: {'stage': 'production', 'accuracy': '0.965'}


## 5. Custom Registry Pattern (PostgreSQL + S3)

For teams that need full control.

In [7]:
# Custom registry: metadata in DB, artifacts in object storage
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional
import json

@dataclass
class ModelVersion:
    name: str
    version: str           # semver: "1.2.3"
    stage: str             # none | staging | production | archived
    artifact_uri: str      # s3://bucket/models/name/version/model.pkl
    run_id: str
    metrics: dict = field(default_factory=dict)
    params: dict = field(default_factory=dict)
    tags: dict = field(default_factory=dict)
    created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())
    description: str = ""

class SimpleModelRegistry:
    """In-memory registry (swap dict for PostgreSQL in production)."""

    def __init__(self):
        self._store: dict[str, list[ModelVersion]] = {}

    def register(self, mv: ModelVersion) -> ModelVersion:
        self._store.setdefault(mv.name, []).append(mv)
        return mv

    def get_latest(self, name: str, stage: str = "production") -> Optional[ModelVersion]:
        versions = [v for v in self._store.get(name, []) if v.stage == stage]
        return versions[-1] if versions else None

    def transition(self, name: str, version: str, new_stage: str):
        for mv in self._store.get(name, []):
            if mv.version == version:
                mv.stage = new_stage
                return mv
        raise ValueError(f"Version {version} of {name} not found")

    def list_versions(self, name: str) -> list:
        return self._store.get(name, [])


# Usage
registry = SimpleModelRegistry()

mv = registry.register(ModelVersion(
    name="fraud-detector",
    version="1.0.0",
    stage="staging",
    artifact_uri="s3://ml-models/fraud-detector/1.0.0/model.pkl",
    run_id="abc123",
    metrics={"auc": 0.97, "f1": 0.91},
    params={"n_estimators": 200},
    description="XGBoost fraud detector v1"
))

registry.transition("fraud-detector", "1.0.0", "production")
champion = registry.get_latest("fraud-detector", stage="production")
print(f"Champion: {champion.name} v{champion.version} | AUC: {champion.metrics['auc']}")

Champion: fraud-detector v1.0.0 | AUC: 0.97


/tmp/ipykernel_158994/147454005.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())


## 6. Model Promotion Workflow with Automated Gates

```
Train → Unit Tests → Staging → Integration Tests → Shadow Mode → Production
                        │                │
                    Accuracy >      Latency < 100ms
                    baseline        No regressions
```

In [8]:
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

def promotion_gate(
    candidate_model,
    champion_model,
    X_test, y_test,
    min_accuracy: float = 0.90,
    min_improvement: float = 0.005
) -> dict:
    """Automated promotion gate returns verdict and metrics."""
    cand_acc = accuracy_score(y_test, candidate_model.predict(X_test))
    champ_acc = accuracy_score(y_test, champion_model.predict(X_test)) if champion_model else 0.0

    cand_auc = roc_auc_score(y_test, candidate_model.predict_proba(X_test)[:, 1])

    passed = (
        cand_acc >= min_accuracy and
        cand_acc >= champ_acc + min_improvement
    )

    return {
        "promote": passed,
        "candidate_accuracy": cand_acc,
        "champion_accuracy": champ_acc,
        "candidate_auc": cand_auc,
        "delta": cand_acc - champ_acc,
        "reason": "PASS" if passed else f"FAIL: acc={cand_acc:.3f} < required={min_accuracy}"
    }

# Demo
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

champion = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)
candidate = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

result = promotion_gate(candidate, champion, X_test, y_test)
for k, v in result.items():
    print(f"  {k}: {v}")

  promote: False
  candidate_accuracy: 0.958041958041958
  champion_accuracy: 0.972027972027972
  candidate_auc: 0.9943820224719101
  delta: -0.013986013986013957
  reason: FAIL: acc=0.958 < required=0.9


## 7. Model Lineage Tracking

Full traceability: **Dataset → Preprocessing → Training Run → Model Version → Deployment**

In [9]:
import hashlib
import json
from datetime import datetime

def compute_data_hash(X, y) -> str:
    """Deterministic hash of dataset for lineage."""
    import numpy as np
    data_bytes = np.array(X).tobytes() + np.array(y).tobytes()
    return hashlib.sha256(data_bytes).hexdigest()[:16]

@dataclass
class ModelLineage:
    model_name: str
    model_version: str
    data_hash: str           # sha256 of training data
    data_source: str         # path or URI
    training_run_id: str
    git_commit: str          # code version
    training_params: dict
    evaluation_metrics: dict
    created_by: str
    created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())

    def to_json(self) -> str:
        return json.dumps(self.__dict__, indent=2)

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
X, y = load_breast_cancer(return_X_y=True)

lineage = ModelLineage(
    model_name="breast-cancer-classifier",
    model_version="1.2.0",
    data_hash=compute_data_hash(X, y),
    data_source="s3://data-lake/breast-cancer/v3/train.parquet",
    training_run_id="run_abc123",
    git_commit="a1b2c3d4",
    training_params={"n_estimators": 100, "max_depth": 5},
    evaluation_metrics={"accuracy": 0.965, "auc": 0.991},
    created_by="ml-pipeline-bot"
)
print(lineage.to_json())

{
  "model_name": "breast-cancer-classifier",
  "model_version": "1.2.0",
  "data_hash": "25e2eb7b7a8745b1",
  "data_source": "s3://data-lake/breast-cancer/v3/train.parquet",
  "training_run_id": "run_abc123",
  "git_commit": "a1b2c3d4",
  "training_params": {
    "n_estimators": 100,
    "max_depth": 5
  },
  "evaluation_metrics": {
    "accuracy": 0.965,
    "auc": 0.991
  },
  "created_by": "ml-pipeline-bot",
  "created_at": "2026-06-19T12:00:44.293063"
}


/tmp/ipykernel_158994/1276222525.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())


## Additional Learning Resources

### Documentation
- **MLflow Model Registry**: https://mlflow.org/docs/latest/model-registry.html
- **BentoML Model Store**: https://docs.bentoml.com/en/latest/concepts/model.html
- **Hugging Face Hub**: https://huggingface.co/docs/hub/index
- **W&B Model Registry**: https://docs.wandb.ai/guides/model_registry

### Papers & Articles
- **Sculley et al. Hidden Technical Debt in ML Systems**: https://papers.nips.cc/paper/2015/file/86df7dcfd896fcaf2674f757a2463eba-Paper.pdf
- **Lessons from the MLOps Trenches**: https://mlops.community/
- **Feature Stores for ML**: https://www.featurestore.org/

### Courses
- **Made With ML MLOps**: https://madewithml.com/#mlops
- **Full Stack Deep Learning**: https://fullstackdeeplearning.com/
- **MLOps Zoomcamp**: https://github.com/DataTalksClub/mlops-zoomcamp